In [ ]:
%pip install pandas matplotlib scikit-learn scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.signal import butter, filtfilt, find_peaks

# 1. Leer y parsear el archivo CSV
filename = './Dataset/csi2minan.csv'
csi_records = []

print("Procesando archivo CSV...")
with open(filename, 'r') as f:
    for line in f:
        if line.startswith('CSI_DATA'):
            # Extraer el arreglo de datos dentro de los corchetes
            match = re.search(r'\"\[(.*?)\]\"', line)
            if match:
                array_str = match.group(1)
                try:
                    vals = [int(x) for x in array_str.split(',')]
                    csi_records.append(vals)
                except:
                    continue

# Alinear la longitud de los arreglos por seguridad
min_len = min(len(r) for r in csi_records)
csi_matrix = np.array([r[:min_len] for r in csi_records])

# 2. Calcular la Amplitud (I/Q)
I = csi_matrix[:, 0::2]
Q = csi_matrix[:, 1::2]
amplitude = np.sqrt(I**2 + Q**2)

# Suavizar ruido con media móvil
window_size = 30 
smoothed_amp = pd.DataFrame(amplitude).rolling(window=window_size, min_periods=1).mean().values

# 3. Reducción de Dimensionalidad (PCA)
scaler = StandardScaler()
amp_scaled = scaler.fit_transform(smoothed_amp)

pca = PCA(n_components=1)
pc1 = pca.fit_transform(amp_scaled).flatten()

# 4. Filtrado Pasa-Banda (Frecuencia de respiración humana)
# Asumimos una tasa de muestreo (fs) aproximada de 100 Hz. Ajusta esto según tu ESP32.
fs = 100 
lowcut = 0.15  # 9 respiraciones por minuto
highcut = 0.4 # 36 respiraciones por minuto
nyq = 0.5 * fs

b, a = butter(2, [lowcut/nyq, highcut/nyq], btype='band')
filtered_signal = filtfilt(b, a, pc1)

# 5. Detección de Picos Estricta
# Calculamos un umbral dinámico basado en la desviación estándar de la señal.
# Esto asegura que solo tomemos los picos que realmente destacan.
signal_std = np.std(filtered_signal)
min_prominence = signal_std * 0.8 # Exigimos que el pico sobresalga al menos este valor

# Mantenemos la distancia mínima (ej. 1.5 segundos)
min_distance = int(3 * fs) 

# Agregamos la prominencia a la función
peaks, _ = find_peaks(filtered_signal, distance=min_distance, prominence=min_prominence)

# Calcular RPM basado en el tiempo promedio entre picos
if len(peaks) > 1:
    peak_intervals = np.diff(peaks) / fs # Intervalos en segundos
    avg_interval = np.mean(peak_intervals)
    rpm = 60 / avg_interval
    print(f"\n--- Resultados ---")
    print(f"Total de picos detectados: {len(peaks)}")
    print(f"Frecuencia Respiratoria Estimada: {rpm:.2f} RPM")
else:
    print("No se detectaron suficientes picos para calcular las RPM.")

# 6. Visualización final
plt.figure(figsize=(12, 5))
plt.plot(np.arange(len(filtered_signal))/fs, filtered_signal, label='Señal Filtrada (PCA)')
plt.plot(peaks/fs, filtered_signal[peaks], "x", color='red', label='Exhalaciones (Picos)')
plt.title('Detección de Respiración vía Wi-Fi CSI')
plt.xlabel('Tiempo estimado (Segundos)')
plt.ylabel('Amplitud Relativa')
plt.legend()
plt.grid(True)
plt.show()


  Using cached matplotlib-3.11.0-cp312-cp312-win_amd64.whl.metadata (80 kB)
Using cached matplotlib-3.11.0-cp312-cp312-win_amd64.whl (9.3 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Procesando archivo CSV...


FileNotFoundError: [Errno 2] No such file or directory: 'csi2minan.csv'